In [25]:
import numpy as np
import jax.numpy as jnp
import jax
import equinox as eqx
import matplotlib as mpl
import matplotlib.pyplot as plt
import pickle
import immunowave as iw
from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
import diffrax as dx
from diffrax import SaveAt
import math
import pickle
import os
jax.config.update("jax_enable_x64", True)

In [26]:
linewidth = 3
fontsize = 24
markersize = 12
markeredgewidth = 2
rc_params["axes.linewidth"] = linewidth
rc_params["font.size"] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams["pdf.fonttype"] = 42
sim_color = np.array([94, 45, 144]) / 255

In [27]:
class State(iw.State):
    u: iw.ScalarField


Here we simulate the dimensionful model to verify the results of dimensional analysis
$$
\partial_t u = D\nabla^2 u + k\frac{u^n}{K^n_D + u^n} - \gamma u + 2I \delta(\vec{x})
$$

with $I, K_D > 0$, and $n > 1$.


In [28]:
class Model(iw.Model):
    KD: float
    n: float
    I: float
    k: float = 1.0
    γ: float = 1.0
    D: float = 1.0
    # Bandwidth adjustment for Dirac delta approximation (grid units)
    bw_adjust: float = 0.5

    def f(self, u):
        KD, n, k, γ = self.KD, self.n, self.k, self.γ
        return k * u.hill(KD, n) - γ * u

    def delta(self, u):
        """Helper function to approximate a Dirac delta as a Gaussian on the same grid as u."""
        σ = self.bw_adjust * u.h

        def gaussian(*coords):
            squared_dist = sum(coord**2 for coord in coords)
            return jnp.exp(-squared_dist / (2 * σ**2))

        result = iw.ScalarField(u.values.shape, u.lb, u.h, fn=gaussian)
        result /= result.integral()
        return result

    def __call__(self, t, state: State, args=None):
        u = state.u
        Δu = u.laplacian(bc="neumann")
        f = self.f
        I = self.I
        δ = self.delta(u)
        D = self.D

        dudt = D * Δu + f(u) + 2 * I * δ

        return State(dudt)
    
class Model_mixed(iw.Model):
    KD: float
    n: float
    I: float
    k: float = 1.0
    γ: float = 1.0
    # Bandwidth adjustment for Dirac delta approximation (grid units)
    bw_adjust: float = 0.5

    def f(self, u):
        KD, n, k, γ = self.KD, self.n, self.k, self.γ
        return k * u.hill(KD, n) - γ * u

    def __call__(self, t, state: State, args=None):
        u = state.u
        f = self.f
        I = self.I

        dudt = f(u) + 2 * I

        return State(dudt)

In [29]:
"""theory"""
def u1(KD, n, k=1.0, γ=1.0):
    return (KD * γ / k) ** (n / (n - 1))


def Ic(KD, n, k=1.0, D=1.0, γ=1.0, d=1):
    return ((n - 1) / (n + 1)) ** 0.5 * u1(KD * γ / k, n) * (D / γ) ** (d / 2) * k


def Ic_mixed(KD, n, k=1.0, γ=1.0):
    return 0.5 * (1 - 1 / n) * (1 / n) ** (1 / (n - 1)) * (KD * γ / k) ** (n / (n - 1))

In [30]:
# KD = 0.01
# n = 4.0
# k = 1.0
# D = 1.0
# bw_adjust = 0.5
# γ = 1.0

# real units
n = 4.0
D = 600.0   # um^2/min
bw_adjust = 0.5
γ = 0.23    # 1/min
k = 1.0
KD = 0.01 * k / γ   


# grid parameters
L = 20 * np.sqrt(D / γ)
n_spatial = 200
h = L / (n_spatial - 1)

# optimization parameter
tolerance = 1e-4
model_low_factor = 0.9
model_high_factor = 1.1

In [31]:
t0 = 0.0
t1 = jnp.inf


@jax.jit
def norm(state: State):
    d = state.u.ndim
    L = state.u.ub[0] - state.u.lb[0]  # Length of the domain in first dimension
    # indicator function for middle of the domain
    ind = iw.ScalarField(
        state.u.values.shape,
        state.u.lb,
        state.u.h,
        fn=lambda *coords: jnp.where(
            sum(abs(coord) ** d for coord in coords) ** (1 / d) < L / 4, 1.0, 0.0
        ),
    )
    return (ind * state.u).integral() / ind.integral()


steady_state_event = dx.Event(dx.steady_state_event(atol=0, rtol=1e-3, norm=norm))

@jax.jit
def mixed_val(state: State):
    return state.u.values[0]


steady_state_event_mixed = dx.Event(dx.steady_state_event(atol=0, rtol=1e-3, norm=mixed_val))

In [32]:
kwargs = dict(
    dt0=1e-4,
    max_steps=10000000,
    atol=1e-8,
    rtol=1e-8,
    throw=True,
    event=steady_state_event,
)

In [33]:
@jax.jit
def mean_activation(model, state):
    steady_state = iw.solve(model, state, 0, jnp.inf, **kwargs).ys.u.map(jnp.squeeze)
    area = math.prod(ub - lb for ub, lb in zip(state.u.ub, state.u.lb))
    return steady_state.integral() / area

In [34]:
def find_Ic(model, state, tolerance=1e-4, max_iterations=20, verbose=False):
    _print = print if verbose else lambda *args, **kwargs: None
    # initial bounds
    model_low = eqx.tree_at(lambda m: m.I, model, model_low_factor * model.I)
    model_high = eqx.tree_at(lambda m: m.I, model, model_high_factor * model.I)
    # If low bound doesn't give low activation, reduce it
    while mean_activation(model_low, state) > 0.5:
        _print("decreasing invalid lower bound")
        model_low = eqx.tree_at(lambda m: m.I, model_low, model_low.I / 2)
    # If high bound doesn't give high activation, increase it
    while mean_activation(model_high, state) < 0.5:
        _print("increasing invalid upper bound")
        model_high = eqx.tree_at(lambda m: m.I, model_high, 2 * model_high.I)

    # Iteratively bisect bounding interval
    for i in range(max_iterations):
        _print(f"iter {i}: [{model_low.I:.3e}, {model_high.I:.3e}]")
        model_mid = eqx.tree_at(
            lambda m: m.I, model_low, (model_low.I + model_high.I) / 2
        )
        if (model_high.I - model_low.I) / model_mid.I < tolerance:
            _print(f"tolerance satisifed: Ic = {model_mid.I:.3e}")
            return model_mid
        if mean_activation(model_mid, state) < 0.5:
            model_low = eqx.tree_at(lambda m: m.I, model, model_mid.I)
        else:
            model_high = eqx.tree_at(lambda m: m.I, model, model_mid.I)

    raise RuntimeError("max iterations reached")
    return model_mid

In [35]:
def D_sweep(state, savedir=None, D_grid=jnp.geomspace(0.4, 4, 6), initial_guesses=None):
    if savedir is not None:
        if not os.path.exists(savedir):
            os.mkdir(savedir)
            
        if os.path.exists(savedir + "/D_sweep.pkl"):
            raise ValueError("file exists! aborting")
            
    # initial guess is a given factor times the 1D prediction
    if initial_guesses is None:
        initial_guesses = Ic(KD, n, k=k, γ=γ, D=D_grid)
    models = [Model(KD=KD, n=n, I=initial_guess, γ=γ, D=D) for D, initial_guess in zip(D_grid, initial_guesses)]
    Ic_grid = jnp.array([find_Ic(model, state, verbose=True, tolerance=tolerance).I for model in models])

    if savedir is not None:
        with open(savedir + "/D_sweep.pkl", "wb") as file:
            save_dict = {"D_grid": D_grid, "Ic_grid": Ic_grid, "models": models}
            pickle.dump(save_dict, file)

    return D_grid, Ic_grid, models

In [36]:
D_grid = jnp.geomspace(60, 6000, 6)

## d=1

In [17]:
savedir = r"/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/1D"


lb = [-L / 2]
shape = (n_spatial,)
state = State(u=iw.ScalarField(shape, lb, h, 0))
tolerance = 1e-4
D_grid, Ic_grid, models = D_sweep(state, savedir=savedir, D_grid=D_grid)

iter 0: [2.426e-02, 2.965e-02]
iter 1: [2.695e-02, 2.965e-02]
iter 2: [2.695e-02, 2.830e-02]
iter 3: [2.695e-02, 2.763e-02]
iter 4: [2.729e-02, 2.763e-02]
iter 5: [2.746e-02, 2.763e-02]
iter 6: [2.746e-02, 2.754e-02]
iter 7: [2.750e-02, 2.754e-02]
iter 8: [2.750e-02, 2.752e-02]
iter 9: [2.750e-02, 2.751e-02]
iter 10: [2.751e-02, 2.751e-02]
iter 11: [2.751e-02, 2.751e-02]
tolerance satisifed: Ic = 2.751e-02
iter 0: [3.845e-02, 4.699e-02]
iter 1: [4.272e-02, 4.699e-02]
iter 2: [4.272e-02, 4.485e-02]
iter 3: [4.272e-02, 4.379e-02]
iter 4: [4.272e-02, 4.325e-02]
iter 5: [4.299e-02, 4.325e-02]
iter 6: [4.312e-02, 4.325e-02]
iter 7: [4.312e-02, 4.319e-02]
iter 8: [4.315e-02, 4.319e-02]
iter 9: [4.317e-02, 4.319e-02]
iter 10: [4.317e-02, 4.318e-02]
iter 11: [4.317e-02, 4.318e-02]
tolerance satisifed: Ic = 4.318e-02
iter 0: [6.093e-02, 7.448e-02]
iter 1: [6.770e-02, 7.448e-02]
iter 2: [6.770e-02, 7.109e-02]
iter 3: [6.770e-02, 6.940e-02]
iter 4: [6.770e-02, 6.855e-02]
iter 5: [6.813e-02, 6.855

## d=2

In [14]:
savedir = r"/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/2D"
# with open(r'/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/1D/D_sweep.pkl', 'rb') as file:
#     Ic_1d = pickle.load(file)['Ic_grid']
# initial_guesses = 2.5 * np.sqrt(D_grid / γ / bw_adjust / h) * Ic_1d
# use our already measured prefactor h(4) ~ 2:
initial_guesses = 2 * (KD / (k / γ)) ** (n / (n - 1)) * k * (D_grid / γ / (bw_adjust * h) / (bw_adjust * h))
lb = [-L / 2, -L / 2]
n_spatial = 200 
shape = (n_spatial, n_spatial)
h = L / (n_spatial - 1)
#initial_guess_factor = 100
model_low_factor = 0.3
model_high_factor = 3
state = State(u=iw.ScalarField(shape, lb, h, 0))
tolerance = 0.5
D_grid, Ic_grid, models = D_sweep(state, savedir=savedir, D_grid=D_grid, initial_guesses=initial_guesses)

increasing invalid upper bound
iter 0: [5.119e-02, 1.024e+00]
iter 1: [5.375e-01, 1.024e+00]
iter 2: [7.807e-01, 1.024e+00]
tolerance satisifed: Ic = 9.022e-01
increasing invalid upper bound
iter 0: [1.286e-01, 2.572e+00]
iter 1: [1.350e+00, 2.572e+00]
iter 2: [1.961e+00, 2.572e+00]
tolerance satisifed: Ic = 2.266e+00
increasing invalid upper bound
iter 0: [3.230e-01, 6.460e+00]
iter 1: [3.391e+00, 6.460e+00]
iter 2: [4.926e+00, 6.460e+00]
tolerance satisifed: Ic = 5.693e+00
increasing invalid upper bound
iter 0: [8.113e-01, 1.623e+01]
iter 1: [8.519e+00, 1.623e+01]
iter 2: [1.237e+01, 1.623e+01]
tolerance satisifed: Ic = 1.430e+01
increasing invalid upper bound
iter 0: [2.038e+00, 4.076e+01]
iter 1: [2.140e+01, 4.076e+01]
iter 2: [3.108e+01, 4.076e+01]
tolerance satisifed: Ic = 3.592e+01
increasing invalid upper bound
iter 0: [5.119e+00, 1.024e+02]
iter 1: [5.375e+01, 1.024e+02]
iter 2: [7.807e+01, 1.024e+02]
tolerance satisifed: Ic = 9.022e+01


In [37]:
with open(r'/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/2D/D_sweep_tol0pt5.pkl', 'rb') as file:
    initial_guesses = pickle.load(file)['Ic_grid']
initial_guesses

Array([ 0.9022354 ,  2.26631285,  5.69272051, 14.29946741, 35.91863816,
       90.22353983], dtype=float64)

In [38]:
"""refining d=2"""
savedir = r"/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/2D"
with open(r'/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/2D/D_sweep_tol0pt5.pkl', 'rb') as file:
    initial_guesses = pickle.load(file)['Ic_grid']
lb = [-L / 2, -L / 2]
n_spatial = 200 
shape = (n_spatial, n_spatial)
h = L / (n_spatial - 1)
#initial_guess_factor = 100
model_low_factor = 0.9
model_high_factor = 1.1
state = State(u=iw.ScalarField(shape, lb, h, 0))
tolerance = 1e-2
D_grid, Ic_grid, models = D_sweep(state, savedir=savedir, D_grid=D_grid, initial_guesses=initial_guesses)

increasing invalid upper bound
iter 0: [8.120e-01, 1.985e+00]
iter 1: [8.120e-01, 1.398e+00]
iter 2: [8.120e-01, 1.105e+00]
iter 3: [9.586e-01, 1.105e+00]
iter 4: [9.586e-01, 1.032e+00]
iter 5: [9.953e-01, 1.032e+00]
iter 6: [1.014e+00, 1.032e+00]
iter 7: [1.014e+00, 1.023e+00]
tolerance satisifed: Ic = 1.018e+00
iter 0: [2.040e+00, 2.493e+00]
iter 1: [2.266e+00, 2.493e+00]
iter 2: [2.380e+00, 2.493e+00]
iter 3: [2.380e+00, 2.436e+00]
iter 4: [2.380e+00, 2.408e+00]
iter 5: [2.394e+00, 2.408e+00]
tolerance satisifed: Ic = 2.401e+00
iter 0: [5.123e+00, 6.262e+00]
iter 1: [5.693e+00, 6.262e+00]
iter 2: [5.693e+00, 5.977e+00]
iter 3: [5.835e+00, 5.977e+00]
iter 4: [5.835e+00, 5.906e+00]
iter 5: [5.835e+00, 5.871e+00]
tolerance satisifed: Ic = 5.853e+00
iter 0: [1.287e+01, 1.573e+01]
iter 1: [1.430e+01, 1.573e+01]
iter 2: [1.430e+01, 1.501e+01]
iter 3: [1.430e+01, 1.466e+01]
iter 4: [1.430e+01, 1.448e+01]
iter 5: [1.439e+01, 1.448e+01]
tolerance satisifed: Ic = 1.443e+01
iter 0: [3.233e+01,

## d=3

In [15]:
savedir = r"/home/brandon/Documents/Code/immunowave/data/2026_01_10_D_sweep/3D"

lb = [-L / 2, -L / 2, -L / 2]
n_spatial = 200  
shape = (n_spatial, n_spatial, n_spatial)
h = L / (n_spatial - 1)

# use our predicting with a fudge factor based on the 2d sims
initial_guesses = 9 * (KD / (k / γ)) ** (n / (n - 1)) * k * (D_grid / γ / (bw_adjust * h) / (bw_adjust * h)) ** (3 / 2)

state = State(u=iw.ScalarField(shape, lb, h, 0))
model_low_factor = 0.3
model_high_factor = 3
tolerance = 1.0   # decreasing tolerance 
D_grid, Ic_grid, models = D_sweep(state, savedir=savedir, D_grid=D_grid, initial_guesses=initial_guesses)

increasing invalid upper bound
iter 0: [1.450e+00, 2.899e+01]
iter 1: [1.522e+01, 2.899e+01]
tolerance satisifed: Ic = 2.211e+01
increasing invalid upper bound
iter 0: [5.771e+00, 1.154e+02]
iter 1: [6.060e+01, 1.154e+02]
tolerance satisifed: Ic = 8.801e+01
iter 0: [2.298e+01, 2.298e+02]
iter 1: [1.264e+02, 2.298e+02]
tolerance satisifed: Ic = 1.781e+02
iter 0: [9.147e+01, 9.147e+02]
iter 1: [5.031e+02, 9.147e+02]
tolerance satisifed: Ic = 7.089e+02
iter 0: [3.641e+02, 3.641e+03]
iter 1: [2.003e+03, 3.641e+03]
tolerance satisifed: Ic = 2.822e+03
iter 0: [1.450e+03, 1.450e+04]
iter 1: [7.973e+03, 1.450e+04]
tolerance satisifed: Ic = 1.123e+04
